In [ ]:
import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
import seaborn as sns
from shapely.geometry import Point

df2012 = pd.read_csv("Crimes2012.csv")
df2013 = pd.read_csv("Crimes2013.csv")
df2014 = pd.read_csv("Crimes2014.csv")

crime_df = pd.concat([df2012, df2013, df2014], ignore_index=True)

crime_df.head()

In [ ]:
columns_needed = [
    "ID",
    "Date",
    "Primary Type",
    "Community Area"
]

crime_df = crime_df[columns_needed]

In [ ]:
crime_df["Date"] = pd.to_datetime(crime_df["Date"], errors="coerce")

crime_df = crime_df.dropna(subset=["Community Area"])
crime_df = crime_df[crime_df["Community Area"] != 0]

crime_df["Community Area"] = crime_df["Community Area"].astype(int)

crime_df["Year"] = crime_df["Date"].dt.year

In [ ]:
crime_area = crime_df.groupby("Community Area").size().reset_index(name="Crime_Count")

crime_area.head()

In [ ]:
import numpy as np

elem_df = pd.read_csv("Chicago_Public_Schools_–_Elementary_School_Progress_Report_Card_(2012-2013)_20260303.csv")
hs_df = pd.read_csv("Chicago_Public_Schools_–_High_School_Progress_Report_Card_(2012-2013)_20260303.csv")

In [ ]:
community_gdf = gpd.read_file("chicago_community_boundaries.json")
community_gdf['area_numbe'] = community_gdf['area_numbe'].astype(int)

community_gdf.head()

In [ ]:
def clean_numeric(series):
    return pd.to_numeric(
        series.astype(str)
        .str.replace('%','',regex=False)
        .str.replace(',','',regex=False)
        .replace('NDA',np.nan),
        errors='coerce'
    )

In [ ]:
elem_df["Reading_Perf"] = clean_numeric(elem_df["Reading Performance All Grades"])
elem_df["Academic_Rank"] = elem_df["Reading_Perf"].rank(pct=True)*100

hs_df["ACT_Score"] = clean_numeric(hs_df["Average Score ACT 2012"])
hs_df["Academic_Rank"] = hs_df["ACT_Score"].rank(pct=True)*100

In [ ]:
for df in [elem_df, hs_df]:

    df["Student_Attendance"] = clean_numeric(df["Student Attendance 2012 - Percent"])
    df["Teacher_Attendance"] = clean_numeric(df["Teacher Attendance 2012 - Percent"])
    df["Misconduct_Rate"] = clean_numeric(df["Misconducts Resulting in Suspensions 2012 - Percent"])
    df["Days_Suspended"] = clean_numeric(df["Average Days of Suspension 2012"])

In [ ]:
for df in [elem_df, hs_df]:

    df["True_Lon"] = pd.to_numeric(df["Latitude"], errors="coerce")
    df["True_Lat"] = pd.to_numeric(df["Longitude"], errors="coerce")

In [ ]:
common_cols = [
    "School ID",
    "School Name",
    "Student_Attendance",
    "Teacher_Attendance",
    "Misconduct_Rate",
    "Days_Suspended",
    "Academic_Rank",
    "True_Lat",
    "True_Lon"
]

comb_df = pd.concat([
    elem_df[common_cols],
    hs_df[common_cols]
], ignore_index=True)

comb_df = comb_df.dropna(subset=["True_Lat","True_Lon"])

In [ ]:
from shapely.geometry import Point

geometry = [Point(xy) for xy in zip(comb_df["True_Lon"], comb_df["True_Lat"])]

schools_gdf = gpd.GeoDataFrame(
    comb_df,
    geometry=geometry,
    crs="EPSG:4326"
)

In [ ]:
mapped_schools = gpd.sjoin(
    schools_gdf,
    community_gdf,
    how="inner",
    predicate="intersects"
)

In [ ]:
mapped_schools.head()

In [ ]:
edu_area = mapped_schools.groupby("area_numbe").agg({
    "Academic_Rank":"mean",
    "Student_Attendance":"mean",
    "Teacher_Attendance":"mean",
    "Misconduct_Rate":"mean",
    "Days_Suspended":"mean"
}).reset_index()

edu_area.head()

In [ ]:
crime_area = crime_df.groupby("Community Area").size().reset_index(name="Crime_Count")

merged = crime_area.merge(
    edu_area,
    left_on="Community Area",
    right_on="area_numbe"
)

In [ ]:
merged = crime_area.merge(
    edu_area,
    left_on="Community Area",
    right_on="area_numbe",
    how="inner"
)

print(merged.shape)
merged.head()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(10,6))
sns.regplot(
    data=merged,
    x="Academic_Rank",
    y="Crime_Count",
    scatter_kws={"alpha":0.6}
)
plt.title("Crime Count (2012–2014) vs Academic Rank (Community Area)")
plt.xlabel("Average Academic Rank Percentile")
plt.ylabel("Crime Count (2012–2014)")
plt.tight_layout()
plt.show()

This scatter plot shows the relationship between crime counts and average academic rank across Chicago community areas. The regression line has a negative slope, which suggests that areas with higher academic performance tend to have lower crime counts. However, the points are widely spread, meaning the relationship is not very strong. Some areas with moderate academic rankings still have relatively high crime levels. This indicates that education performance may be related to crime, but it is likely not the only factor influencing crime patterns.

In [ ]:
corr_cols = ["Crime_Count","Academic_Rank","Student_Attendance","Teacher_Attendance","Misconduct_Rate","Days_Suspended"]
corr = merged[corr_cols].corr()

plt.figure(figsize=(9,7))
sns.heatmap(corr, annot=True, vmin=-1, vmax=1, fmt=".2f", cmap="coolwarm")
plt.title("Correlation Matrix: Crime and Education Factors")
plt.tight_layout()
plt.show()

The correlation matrix helps summarize the relationships between crime and several education indicators. Crime count has a moderate negative correlation with academic rank (-0.38) and student attendance (-0.33). This suggests that communities with better academic performance and higher student attendance tend to have fewer crimes. Teacher attendance shows almost no relationship with crime (-0.04). Misconduct rate and days suspended show small positive correlations with crime, meaning areas with more disciplinary issues may experience slightly higher crime levels. Overall, the relationships exist but are not very strong.

In [ ]:
import numpy as np

# Define thresholds
crime_threshold = merged["Crime_Count"].quantile(0.75)   # top 25% crime
rank_threshold = merged["Academic_Rank"].quantile(0.75)  # top 25% school rank

# Identify exceptional areas
merged["High_Crime"] = merged["Crime_Count"] >= crime_threshold
merged["High_Rank"] = merged["Academic_Rank"] >= rank_threshold

exception_areas = merged[(merged["High_Crime"]) & (merged["High_Rank"])]

exception_areas

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(10,6))

sns.scatterplot(
    data=merged,
    x="Academic_Rank",
    y="Crime_Count",
    color="lightgray"
)

# highlight exception areas
sns.scatterplot(
    data=exception_areas,
    x="Academic_Rank",
    y="Crime_Count",
    color="red",
    s=120,
    label="High Crime + Strong Schools"
)

plt.title("High Crime Areas with Strong Academic Performance")
plt.xlabel("Academic Rank Percentile")
plt.ylabel("Crime Count (2012–2014)")
plt.legend()

plt.show()

In [ ]:
exception_areas[["Community Area","Crime_Count","Academic_Rank"]].sort_values(
    "Crime_Count", ascending=False
)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

top10 = merged.head(10)

g = sns.lmplot(
    data=top10,
    x="Academic_Rank",
    y="Crime_Count",
    col="Community Area",
    col_wrap=5,
    height=4,
    scatter_kws={"alpha":0.7}
)

g.fig.suptitle("Crime vs Education Across Chicago Neighborhoods", y=1.02)

plt.show()